In [1]:
import pandas as pd
import numpy as np
from IPython.display import Image, HTML

In [41]:
df = pd.read_csv('amazon_specs_CLEANED.csv')

In [3]:
df['category'].value_counts()

category
Office, Printing & Power        587
Accessories                     217
Audio & Media Systems           210
Peripherals & Input             161
PC Components (Core)            135
Networking & Smart Home         114
Out of Scope                    111
Displays & Mounting             105
Cameras, Photography & Video    103
Computers & Gaming               95
Mobile Devices                   91
ERROR_API_FAIL                    1
Name: count, dtype: int64

In [42]:
df[df['category'] == 'Computers & Gaming']['subtype'].value_counts()

subtype
laptop                59
desktop               11
vr_headset             4
excluded               3
console                2
handheld_gaming        2
mini_pc                2
nas                    2
video_capture_card     1
all_in_one             1
case                   1
Name: count, dtype: int64

In [45]:
import pandas as pd


# 1. Carga tus datos (ajusta los nombres de tus archivos)
df_specs = pd.read_csv('amazon_specs_CLEANED.csv')
df_prices = pd.read_csv('../../Datasets/amazon_final.csv')

# 2. Une ambos datasets (usando el ID o el Título como llave)
df_final = pd.merge(
    df_specs, 
    df_prices, 
    left_on='original_title', 
    right_on='product_title'
)

# 3. Revertir el log1 para ver el precio real
# Si usaste log natural (ln(1+x)):
df_final['price_real'] = np.exp(df_final['log_original_price']) - 1

# Si usaste log base 10 (log10(1+x)):
# df['price_real'] = (10 ** df['log1_price']) - 1

# 4. Filtrar por la categoría Accesorios y ordenar por precio de mayor a menor
expensive_accessories = df_final[df_final['category'] == 'Computers & Gaming'].sort_values(by='price_real', ascending=True).iloc[0:700]
# 5. Mostrar el Top 20
print("AUDITORÍA DE ACCESORIOS CAROS:")

# 1. Definimos una función que envuelve la URL en una etiqueta HTML <img>
def render_image(url):
    return f'<img src="{url}" width="80" >'

# 2. Seleccionamos la muestra
sample_peripherals = expensive_accessories[['original_title', 'subtype', 'price_real', 'product_image_url']]

# 3. Mostramos la tabla renderizando el HTML
# 'escape=False' permite que el navegador lea los tags <img> en lugar de verlos como texto
HTML(sample_peripherals.to_html(escape=False, formatters=dict(product_image_url=render_image)))

AUDITORÍA DE ACCESORIOS CAROS:


,original_title,subtype,price_real,product_image_url
765,NZXT H5 Flow 2024 - Compact ATX Mid-Tower PC Gaming Case - High Airflow - 2 x 120mm Fans Included - 360mm Front & 240mm Top Radiator Support - Cable Management System - Tempered Glass - Black,case,80.99,
1124,"HORI Racing Wheel Apex for Playstation 5, PlayStation 4 and PC - Officially Licensed by Sony - Compatible with Gran Turismo 7",excluded,119.99,
1902,ASUS E410 Intel Celeron N4020 4GB 64GB 14-Inch HD LED Win 10 Laptop (Star Black),laptop,158.86,
741,"ASUS Chromebook CM14 Laptop, 14"" HD Anti-Glare Display (1366x768), MediaTek Kompanio 520, 4GB RAM, 64GB eMMC, ChromeOS, Gray, CM1402CM2A-DS44, Gravity Grey",laptop,162.00,
1776,"HP 2023 Chromebook Laptop, 14 Inch Display, Intel Celeron N4120 Processor, 4GB RAM, 64GB eMMC, Intel UHD Graphics 600, WiFi, Bluetooth, Chrome OS, Modern Gray",laptop,169.00,
727,"Apple Macbook Air 2017 with 1.8GHz Intel Core i5 (13-inch, 8GB RAM, 128GB SSD Storage) (QWERTY English) Silver (Renewed)",laptop,169.00,
1888,"HP 14"" HD Chromebook Laptop for Students, Intel Quad-Core N4120(> N4020), 4GB RAM, 64GB eMMC, WiFi, Webcam, HDMI, USB-A&C, 14 Hours Battery life, ZOOM, Chrome OS, CUE Accessories",laptop,173.95,
179,"HP Chromebook 14 Laptop, Intel Celeron N4120, 4 GB RAM, 64 GB eMMC, 14"" HD Display, Chrome OS, Thin Design, 4K Graphics, Long Battery Life, Ash Gray Keyboard (14a-na0226nr, 2022, Mineral Silver)",laptop,174.06,
368,"Elgato HD60 X - Stream and Record in 1080p60 HDR10 or 4K30 with Ultra-low Latency on PS5|Pro, PS4|Pro, Xbox Series X/S, Xbox One X|S, Nintendo Switch 2, in OBS and More, Works with PC and Mac",video_capture_card,179.99,
1719,Synology 2-Bay DiskStation DS223j (Diskless),nas,194.99,


In [91]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
FILE_PATH = "amazon_specs_CLEANED.csv"

# --- DICCIONARIO DE DATOS ---
# Formato: "Parte única del título": Capacidad en GB
# NOTA: Los CDs suelen ser de 700MB = 0.7 GB
#       Los DVDs son de 4.7 GB
#       Los Blu-Ray son de 25 GB o 50 GB
CD_MAP = {
    # Ejemplo 1: Título exacto o parcial -> 700MB (0.7 GB)
    """Verbatim CD-R Blank Discs 700MB 80 Minutes 52x Recordable Disc for Data and Music - 50 Pack Spindle,Silver""": 0.7,
}

def update_cds_manually():
    print(f"📂 Cargando {FILE_PATH}...")
    
    if not os.path.exists(FILE_PATH):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(FILE_PATH)
    total_modificados = 0

    print("\n💿 Iniciando actualización de CDs...")

    for titulo_parcial, gigas in CD_MAP.items():
        # Buscamos el producto que contenga ese texto (regex=False para evitar problemas con paréntesis)
        mask = df['original_title'].str.contains(titulo_parcial, case=False, regex=False, na=False)
        
        count = mask.sum()
        
        if count > 0:
            # 1. Aplicar cambios
            df.loc[mask, 'subtype'] = 'cd'  # Asignamos el subtipo nuevo
            df.loc[mask, 'storage_gb'] = gigas # Asignamos el almacenamiento manual
            
            # 2. Asegurarnos de que están en Office (por si acaso estaban en otro lado)
            df.loc[mask, 'category'] = 'Office, Printing & Power'
            
            print(f"   ✅ Actualizados {count} registros para: '{titulo_parcial[:30]}...' -> {gigas} GB")
            total_modificados += count
        else:
            print(f"   ⚠️ NO ENCONTRADO: '{titulo_parcial[:30]}...'")

    if total_modificados > 0:
        # Guardar
        df.to_csv(FILE_PATH, index=False)
        print(f"\n💾 Guardado. Se han modificado {total_modificados} filas en total.")
    else:
        print("\n🤷‍♂️ No se realizaron cambios (revisa los títulos).")


update_cds_manually()

📂 Cargando amazon_specs_CLEANED.csv...

💿 Iniciando actualización de CDs...
   ✅ Actualizados 1 registros para: 'Verbatim CD-R Blank Discs 700M...' -> 0.7 GB

💾 Guardado. Se han modificado 1 filas en total.


In [44]:
import os
# --- CONFIGURACIÓN ---
INPUT_FILE = "amazon_specs_CLEANED.csv"  # Tu archivo actual
OUTPUT_FILE = "amazon_specs_CLEANED.csv"   # El archivo nuevo corregido

def fix_airtags():
    print(f"📂 Cargando {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 1. IDENTIFICAR LOS PRODUCTOS A MOVER
    # Creamos las condiciones por separado para que el código sea legible
    condicion_cat = df['category'] == 'Computers & Gaming'
    condicion_sub = df['subtype'] == 'excluded'

    # Pasamos a minúsculas una sola vez para que sea más eficiente
    title_lower = df['original_title'].str.lower()

    # Lógica: Contiene 'memory card' O (contiene 'card' Y contiene 'sdxc')
    condicion_titulo = (title_lower.str.contains('Developer Kit'))

    # Aplicamos los filtros y ordenamos
    filtro = (condicion_cat & condicion_sub & condicion_titulo)
    count = filtro.sum()
    print(f"🔍 Detectados {count} productos mal clasificados como 'mouse' en Accesories.")

    if count == 0:
        print("✅ No hay nada que corregir. Todo parece estar bien.")
        return
    

    # 3. CAMBIAR CATEGORÍA Y SUBTIPO
    print("🏷️ Actualizando etiquetas de categoría y subtipo...")
    df.loc[filtro, 'category'] = 'PC Components (Core)'
    df.loc[filtro, 'subtype'] = 'motherboard'

    # 4. GUARDAR
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 ¡Éxito! Archivo corregido guardado como: {OUTPUT_FILE}")
    
fix_airtags()

📂 Cargando amazon_specs_CLEANED.csv...
🔍 Detectados 0 productos mal clasificados como 'mouse' en Accesories.
✅ No hay nada que corregir. Todo parece estar bien.


In [6]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
FILE_PATH = "amazon_specs_CLEANED.csv"

# Pega aquí los títulos EXACTOS que quieres mover (usa comillas triples para evitar problemas)
TITLES_TO_MOVE = [
    """TP-Link 2.5GB PCIe Network Card (TX201) – PCIe to 2.5 Gigabit Ethernet Network Adapter, Supports Windows 11/10/8.1/8/7, Win Server 2022/2019/2016, Linux""",
    """StarTech.com 2.5GbE USB-C to Ethernet Adapter, NBASE-T NIC, USB 3.0 Type-C 2.5/1G Multi Speed Network, Thunderbolt Compatible""",
    """StarTech.com 2.5GbE USB-C to Ethernet Adapter, 100W PD Pass-Through, NBASE-T NIC, USB 3.0 Type-C 2.5G Multi Speed Network""",
    """TP-Link AV1000 Powerline Ethernet Adapter KIT - Gigabit Port, Plug Pair &Play, Ethernet Over Power, Nano Size, Power Saving Mode, Network Adapter, Free Expert Help (TL-PA7017 KIT)"""
]

def move_to_printer():
    print(f"📂 Cargando {FILE_PATH}...")
    
    if not os.path.exists(FILE_PATH):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(FILE_PATH)
    
    # 1. FILTRAR POR LA LISTA DE TÍTULOS
    # Usamos isin() para buscar si el título está en nuestra lista
    mask = df['original_title'].isin(TITLES_TO_MOVE)
    
    count = mask.sum()
    
    if count == 0:
        print("⚠️ No se encontró ninguno de los títulos indicados.")
        print("Asegúrate de que sean idénticos (espacios, comas, etc).")
        return

    print(f"🔄 Se han encontrado {count} productos. Moviendo a 'Office' -> 'printer'...")

    # 2. ACTUALIZAR CATEGORÍA Y SUBTIPO
    df.loc[mask, 'category'] = 'Networking & Smart Home'
    df.loc[mask, 'subtype'] = 'wifi_adapter'

    # 3. GUARDAR
    df.to_csv(FILE_PATH, index=False)
    
    print(f"✅ ¡Hecho! Archivo actualizado.")
    # Mostramos los cambios para confirmar
    print(df.loc[mask, ['original_title', 'category', 'subtype']])

move_to_printer()

📂 Cargando amazon_specs_CLEANED.csv...
🔄 Se han encontrado 4 productos. Moviendo a 'Office' -> 'printer'...
✅ ¡Hecho! Archivo actualizado.
                                         original_title  \
1104  TP-Link AV1000 Powerline Ethernet Adapter KIT ...   
1309  TP-Link 2.5GB PCIe Network Card (TX201) – PCIe...   
1358  StarTech.com 2.5GbE USB-C to Ethernet Adapter,...   
1359  StarTech.com 2.5GbE USB-C to Ethernet Adapter,...   

                     category       subtype  
1104  Networking & Smart Home  wifi_adapter  
1309  Networking & Smart Home  wifi_adapter  
1358  Networking & Smart Home  wifi_adapter  
1359  Networking & Smart Home  wifi_adapter  


In [10]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
FILE_PATH = "amazon_specs_CLEANED.csv"

# Pega aquí el título EXACTO que quieres borrar (cuidado con espacios al final)
TITLE_TO_REMOVE = 'Skylight Frame – WiFi Digital Picture Frame Customer Support, Touch Screen Digital Photo Frame with Easy Setup, Photo Gifts for Parents and Grandparents - 10 Inch Black'
def delete_by_title_string():
    print(f"📂 Cargando {FILE_PATH}...")
    
    if not os.path.exists(FILE_PATH):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(FILE_PATH)
    initial_count = len(df)
    
    # 1. BUSCAR EL TÍTULO EXACTO
    # Creamos una máscara booleana donde el título coincida exactamente
    mask = df['original_title'] == TITLE_TO_REMOVE
    
    # Obtenemos las filas que coinciden
    rows_to_delete = df[mask]
    
    if rows_to_delete.empty:
        print(f"⚠️ No se encontró ninguna fila con el título exacto:\n'{TITLE_TO_REMOVE}'")
        print("Consejo: Verifica espacios extra o signos de puntuación.")
        return

    # 2. MOSTRAR Y CONFIRMAR
    print(f"\n🗑️ Se han encontrado {len(rows_to_delete)} filas con ese título. Eliminando...")
    
    # Mostramos los índices que se van a borrar por si acaso
    for idx in rows_to_delete.index:
        print(f"   ❌ Borrando índice {idx}...")

    # 3. ELIMINAR (Hacemos drop usando los índices que encontramos)
    df = df.drop(rows_to_delete.index)

    # 4. GUARDAR
    df.to_csv(FILE_PATH, index=False)
    
    print(f"\n✅ ¡Hecho! Archivo actualizado.")
    print(f"📊 Antes: {initial_count} -> Ahora: {len(df)} filas.")


delete_by_title_string()

📂 Cargando amazon_specs_CLEANED.csv...

🗑️ Se han encontrado 1 filas con ese título. Eliminando...
   ❌ Borrando índice 229...

✅ ¡Hecho! Archivo actualizado.
📊 Antes: 1930 -> Ahora: 1929 filas.


In [27]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
FILE_PATH = "amazon_specs_CLEANED.csv"
PHRASE_TO_REMOVE = "Instant Film"

def remove_instant_film_verbose():
    print(f"📂 Cargando {FILE_PATH}...")
    
    if not os.path.exists(FILE_PATH):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(FILE_PATH)
    initial_count = len(df)
    
    # 1. IDENTIFICAR FILAS A BORRAR
    # Buscamos la frase en 'original_title', ignorando mayúsculas
    mask_to_remove = df['original_title'].str.contains(PHRASE_TO_REMOVE, case=False, na=False)
    
    # Obtener los títulos que vamos a borrar
    titulos_a_borrar = df.loc[mask_to_remove, 'original_title'].tolist()
    count_to_remove = len(titulos_a_borrar)
    
    if count_to_remove == 0:
        print(f"✅ No se encontraron productos con '{PHRASE_TO_REMOVE}'.")
        return

    # 2. IMPRIMIR LOS TÍTULOS (LO QUE PEDISTE)
    print(f"\n🗑️ SE VAN A ELIMINAR ESTOS {count_to_remove} PRODUCTOS:")
    print("="*60)
    for i, titulo in enumerate(titulos_a_borrar, 1):
        print(f"{i}. {titulo[:100]}...") # Imprime los primeros 100 caracteres
    print("="*60)

    # 3. FILTRAR Y GUARDAR
    df_cleaned = df[~mask_to_remove]
    df_cleaned.to_csv(FILE_PATH, index=False)
    
    print(f"\n💾 Archivo actualizado guardado en: {FILE_PATH}")
    print(f"📊 Antes: {initial_count} -> Ahora: {len(df_cleaned)} filas.")


remove_instant_film_verbose()

📂 Cargando amazon_specs_CLEANED.csv...

🗑️ SE VAN A ELIMINAR ESTOS 9 PRODUCTOS:
1. Fujifilm Instax Mini Instant Film, 10 Sheets x 5 Packs (Total 50 Shoots)...
2. Fujifilm INSTAX Mini Instant Film 2 Pack = 20 Sheets (White) for Fujifilm Mini 8 & Mini 9 Cameras, M...
3. Fujifilm INSTAX Mini Instant Film (White) for Fujifilm Mini 8,9,11,12 Cameras w/Microfiber Cloth by ...
4. Fujifilm Instax Mini Instant Film, 10 Sheets×5 Pack(Total 50 Shoots)...
5. Fujifilm Instax Wide Instant Film Twin Pack - 20 Exposures...
6. Fujifilm Instax Mini Instant Film (3 Twin Packs, 60 Total Pictures) - International Version...
7. Fujifilm Instax Mini Instant Film, 2 x 10 Shoots X 2Pack (Total 40 Shoots) Value Set...
8. Fujifilm Instax Mini Instant Film,10 Sheets×5 Pack(Total 50 Shoots) Bundled with Tudak Photo Magnet ...
9. Fujifilm Instax Mini Instant Film (10 Twin Packs, 200 Total Pictures) for Instax Cameras...

💾 Archivo actualizado guardado en: amazon_specs_CLEANED.csv
📊 Antes: 1928 -> Ahora: 1919 filas.

In [39]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
INPUT_FILE = "amazon_specs_CLEANED.csv"  # Tu archivo actual
OUTPUT_FILE = "amazon_specs_CLEANED.csv"   # El archivo nuevo corregido

def fix_airpods():
    print(f"📂 Cargando {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 1. IDENTIFICAR LOS PRODUCTOS A MOVER
    filtro_airpods = df.loc[488]
    
    # count = filtro_airpods.sum()
    # print(f"🔍 Detectados {count} productos mal clasificados como 'earbuds' en Mobile Devices.")
    print(filtro_airpods.subtype)

    # if count == 0:
    #     print("✅ No hay nada que corregir. Todo parece estar bien.")
    #     return
    

    # # 3. CAMBIAR CATEGORÍA Y SUBTIPO
    print("🏷️ Actualizando etiquetas de categoría y subtipo...")
    df.loc[488, 'category'] = 'Peripherals & Input'
    df.loc[488, 'subtype'] = 'controller'
    

    # # 4. GUARDAR
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 ¡Éxito! Archivo corregido guardado como: {OUTPUT_FILE}")
    
fix_airpods()

📂 Cargando amazon_specs_CLEANED.csv...
excluded
🏷️ Actualizando etiquetas de categoría y subtipo...
🎉 ¡Éxito! Archivo corregido guardado como: amazon_specs_CLEANED.csv


In [17]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
INPUT_FILE = "amazon_specs_CLEANED.csv"
OUTPUT_FILE = "amazon_specs_CLEANED.csv"

def fix_psu_smart():
    print(f"📂 Cargando {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 1. DEFINICIÓN DE FILTROS LÓGICOS
    
    # A) Debe tener "Power Supply" en el título (Fuente de alimentación)
    has_psu = df['original_title'].str.contains("Power Supply", case=False, na=False)
    
    # B) NO debe tener la palabra "UPS" (Sistema de Alimentación Ininterrumpida)
    # Usamos espacios (" UPS ", "UPS ") o regex boundaries para no confundir con palabras como "SUPER" o "GROUPS"
    # \bUPS\b busca la palabra exacta UPS
    has_ups = df['original_title'].str.contains(r"\bUPS\b", case=False, regex=True, na=False)
    
    # C) NO debe tener "Battery Backup" (otra forma común de llamar a los SAIs)
    has_battery = df['original_title'].str.contains("Battery Backup", case=False, na=False)
    
    # D) Que no esté ya en la categoría correcta
    not_already_core = df['category'] != 'PC Components (Core)'

    # --- LÓGICA FINAL ---
    # Queremos: (Tiene Power Supply) Y (NO es UPS) Y (NO es Batería) Y (No está ya clasificado)
    # El símbolo ~ significa "NO" (negación)
    filtro_final = has_psu & (~has_ups) & (~has_battery) & not_already_core
    
    count = filtro_final.sum()
    print(f"🔍 Detectadas {count} fuentes de alimentación reales (excluyendo UPS/SAIs).")

    if count == 0:
        print("✅ No hay nada que corregir.")
        return

    # 2. LIMPIEZA Y REUBICACIÓN
    print("🧹 Limpiando columnas de UPS (capacity_value y unit) para estos productos...")
    df.loc[filtro_final, 'ups_capacity_value'] = None
    df.loc[filtro_final, 'ups_capacity_unit'] = None

    print("🏷️ Moviendo a 'PC Components (Core)' con subtipo 'psu'...")
    df.loc[filtro_final, 'category'] = 'PC Components (Core)'
    df.loc[filtro_final, 'subtype'] = 'psu'

    # 3. GUARDAR
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 ¡Éxito! Archivo corregido guardado en: {OUTPUT_FILE}")

fix_psu_smart()

📂 Cargando amazon_specs_CLEANED.csv...
🔍 Detectadas 8 fuentes de alimentación reales (excluyendo UPS/SAIs).
🧹 Limpiando columnas de UPS (capacity_value y unit) para estos productos...
🏷️ Moviendo a 'PC Components (Core)' con subtipo 'psu'...
🎉 ¡Éxito! Archivo corregido guardado en: amazon_specs_CLEANED.csv


In [37]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
INPUT_FILE = "amazon_specs_CLEANED.csv"  # Tu archivo actual
OUTPUT_FILE = "amazon_specs_CLEANED.csv"   # El archivo nuevo corregido

def fix_controllers():
    print(f"📂 Cargando {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 1. IDENTIFICAR LOS PRODUCTOS A MOVER
    filtro_controllers = (df['category'] == 'Computers & Gaming') & (df['subtype'] == 'controller')
    
    count = filtro_controllers.sum()
    print(f"🔍 Detectados {count} productos mal clasificados como 'earbuds' en Mobile Devices.")

    if count == 0:
        print("✅ No hay nada que corregir. Todo parece estar bien.")
        return
    

    # 3. CAMBIAR CATEGORÍA Y SUBTIPO
    print("🏷️ Actualizando etiquetas de categoría y subtipo...")
    df.loc[filtro_controllers, 'category'] = 'Peripherals & Input'
    df.loc[filtro_controllers, 'subtype'] = 'controller'
    # 4. GUARDAR
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 ¡Éxito! Archivo corregido guardado como: {OUTPUT_FILE}")
    
fix_controllers()

📂 Cargando amazon_specs_CLEANED.csv...
🔍 Detectados 4 productos mal clasificados como 'earbuds' en Mobile Devices.
🏷️ Actualizando etiquetas de categoría y subtipo...
🎉 ¡Éxito! Archivo corregido guardado como: amazon_specs_CLEANED.csv


In [ ]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
INPUT_FILE = "amazon_specs_CLEANED.csv"  # Tu archivo actual
OUTPUT_FILE = "amazon_specs_CLEANED.csv"   # El archivo nuevo corregido

def fix_controller():
    print(f"📂 Cargando {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 1. IDENTIFICAR LOS PRODUCTOS A MOVER
    # Condición: Categoría es 'Accessories' Y Subtype es 'charger_aux'
    filtro_controller = (df['category'] == 'Computers & Gaming') & (df['subtype'] == 'gamepad')
    
    count = filtro_controller.sum()
    print(f"🔍 Detectados {count} productos mal clasificados como 'adapter' en Networking & Smart Home.")

    if count == 0:
        print("✅ No hay nada que corregir. Todo parece estar bien.")
        return
    

    # 3. CAMBIAR CATEGORÍA Y SUBTIPO
    print("🏷️ Actualizando etiquetas de categoría y subtipo...")
    df.loc[filtro_controller, 'category'] = 'controller'

    # 4. GUARDAR
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 ¡Éxito! Archivo corregido guardado como: {OUTPUT_FILE}")
    
fix_controller()

📂 Cargando amazon_specs_CLEANED.csv...
🔍 Detectados 1 productos mal clasificados como 'adapter' en Networking & Smart Home.
🏷️ Actualizando etiquetas de categoría y subtipo...
🎉 ¡Éxito! Archivo corregido guardado como: amazon_specs_CLEANED.csv


In [70]:
import pandas as pd
import os

# --- CONFIGURACIÓN ---
INPUT_FILE = "amazon_specs_enriched_final.csv"  # Tu archivo actual
OUTPUT_FILE = "amazon_specs_CLEANED.csv"   # El archivo nuevo corregido

def fix_speakerphone():
    print(f"📂 Cargando {INPUT_FILE}...")
    
    if not os.path.exists(INPUT_FILE):
        print("❌ Error: No encuentro el archivo.")
        return

    df = pd.read_csv(INPUT_FILE)
    
    # 1. IDENTIFICAR LOS PRODUCTOS A MOVER
    # Condición: Categoría es 'Accessories' Y Subtype es 'charger_aux'
    filtro_speakerphone = (df['category'] == 'Audio & Media Systems') & (df['subtype'] == 'speakerphone')
    
    count = filtro_speakerphone.sum()
    print(f"🔍 Detectados {count} productos mal clasificados como 'charger_aux' en Accessories.")

    if count == 0:
        print("✅ No hay nada que corregir. Todo parece estar bien.")
        return

    # 2. MOVER DATOS DE COLUMNA (Wattage)
    # Copiamos el valor de 'power_wattage' (donde estaba antes) a 'pd_wattage' (donde debe estar en Office)
    # Solo lo hacemos para las filas que cumplen el filtro
    print("⚡ Migrando vatios de 'power_capacity_value' a 'pd_wattage'...")
    df.loc[filtro_speakerphone, 'pd_wattage'] = df.loc[filtro_speakerphone, 'power_capacity_value']
    

    # 3. CAMBIAR CATEGORÍA Y SUBTIPO
    print("🏷️ Actualizando etiquetas de categoría y subtipo...")
    df.loc[filtro_speakerphone, 'category'] = 'Audio & Media Systems'
    df.loc[filtro_speakerphone, 'subtype'] = 'speaker'

    # --- BONUS: ARREGLAR AIRTAGS (Si quieres activarlo, descomenta estas líneas) ---
    # filtro_airtags = (df['original_title'].str.contains("AirTag", case=False, na=False)) & (df['category'] != 'Networking & Smart Home')
    # if filtro_airtags.sum() > 0:
    #     print(f"🍏 Movindo {filtro_airtags.sum()} AirTags a Networking...")
    #     df.loc[filtro_airtags, 'category'] = 'Networking & Smart Home'
    #     df.loc[filtro_airtags, 'subtype'] = 'smart_sensor'

    # 4. GUARDAR
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 ¡Éxito! Archivo corregido guardado como: {OUTPUT_FILE}")
    print("Ahora tus cargadores tienen la categoría correcta y 'pd_wattage' relleno.")
    
fix_speakerphone()

📂 Cargando amazon_specs_enriched_final.csv...
🔍 Detectados 2 productos mal clasificados como 'charger_aux' en Accessories.
⚡ Migrando vatios de 'power_capacity_value' a 'pd_wattage'...
🏷️ Actualizando etiquetas de categoría y subtipo...
🎉 ¡Éxito! Archivo corregido guardado como: amazon_specs_CLEANED.csv
Ahora tus cargadores tienen la categoría correcta y 'pd_wattage' relleno.


In [17]:
import pandas as pd


# 1. Carga tus datos (ajusta los nombres de tus archivos)
df_specs = pd.read_csv('amazon_specs_CLEANED.csv')
df_prices = pd.read_csv('../../Datasets/amazon_final.csv')

# 2. Une ambos datasets (usando el ID o el Título como llave)
df_final = pd.merge(
    df_specs, 
    df_prices, 
    left_on='original_title', 
    right_on='product_title'
)

# 3. Revertir el log1 para ver el precio real
# Si usaste log natural (ln(1+x)):
df_final['price_real'] = np.exp(df_final['log_original_price']) - 1

# Si usaste log base 10 (log10(1+x)):
# df['price_real'] = (10 ** df['log1_price']) - 1

# 4. Filtrar por la categoría Accesorios y ordenar por precio de mayor a menor
expensive_accessories = df_final[df_final['category'] == 'Mobile Devices'].sort_values(by='price_real', ascending=False).iloc[0:164]
aux = expensive_accessories[expensive_accessories['subtype'] == 'earbuds']
# 5. Mostrar el Top 20
print("AUDITORÍA DE ACCESORIOS CAROS:")




# 1. Definimos una función que envuelve la URL en una etiqueta HTML <img>
def render_image(url):
    return f'<img src="{url}" width="80" >'

# 2. Seleccionamos la muestra
sample_peripherals = aux[['original_title', 'price_real', 'product_image_url']]

# 3. Mostramos la tabla renderizando el HTML
# 'escape=False' permite que el navegador lea los tags <img> en lugar de verlos como texto
HTML(sample_peripherals.to_html(escape=False, formatters=dict(product_image_url=render_image)))

AUDITORÍA DE ACCESORIOS CAROS:


,original_title,price_real,product_image_url


In [25]:
df_cleaned = pd.read_csv('amazon_specs_CLEANED.csv')
df_cleaned['category'].value_counts()

category
Office, Printing & Power        614
Audio & Media Systems           214
Accessories                     190
Peripherals & Input             164
PC Components (Core)            135
Networking & Smart Home         114
Out of Scope                    111
Displays & Mounting             105
Cameras, Photography & Video    103
Computers & Gaming               95
Mobile Devices                   84
ERROR_API_FAIL                    1
Name: count, dtype: int64